# My Own Recommender 🎬🎧
Most recommender exercises predict ratings for *anonymous* users. This one recommends for **you**:
1. **Movies:** rate ~15 films, become user #611 of MovieLens, and get recommendations via matrix factorisation, *without retraining* (fold-in). Then blend in content (genres + tags) so niche Danish films get a chance.
2. **Music:** load your own **Spotify history export** and learn artist embeddings from your listening sessions (word2vec's idea, done with a matrix).

Guide note: [[My Own Recommender]] · background: [[Collaborative Filtering and Matrix Factorization]], [[Content-Based and Hybrid Recommenders]], [[Project - MovieLens Recommender]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import json, pathlib, urllib.request, zipfile, io, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

DATA = pathlib.Path("data/ml-latest-small")
if not DATA.exists():
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip").read())).extractall("data")
ratings = pd.read_csv(DATA / "ratings.csv"); movies = pd.read_csv(DATA / "movies.csv"); tags = pd.read_csv(DATA / "tags.csv")
uid = {u: i for i, u in enumerate(sorted(ratings.userId.unique()))}; iid = {m: i for i, m in enumerate(movies.movieId)}
ratings["u"] = ratings.userId.map(uid); ratings["i"] = ratings.movieId.map(iid)
n_users, n_items = len(uid), len(iid)
title = movies.title.to_numpy()

def find_movie(query, n=8):
    """Search titles (case-insensitive) – use it to fill in MY_RATINGS with exact titles."""
    hits = movies[movies.title.str.contains(query, case=False, regex=False)]
    return hits.assign(n_ratings=hits.movieId.map(ratings.movieId.value_counts()).fillna(0).astype(int))[["title", "genres", "n_ratings"]].head(n)

print(f"{len(ratings):,} ratings, {n_users} users, {n_items:,} movies")
find_movie("pusher")

## 0. Your ratings ✍️
Replace the example with **your** ratings (0.5–5 stars, half stars allowed). Use `find_movie("...")` to get exact titles. ~15–30 ratings work well; include films you disliked too, since that's information. (The examples lean Danish to show off the hybrid part.)

In [ ]:
MY_RATINGS = {
    "Celebration, The (Festen) (1998)": 4.5,
    "Pusher (1996)": 4.0,
    "Flickering Lights (Blinkende lygter) (2000)": 5.0,
    "Adam's Apples (Adams æbler) (2005)": 4.5,
    "Green Butchers, The (Grønne slagtere, De) (2003)": 4.0,
    "In a Better World (Hævnen) (2010)": 4.0,
    "Matrix, The (1999)": 4.5,
    "Pulp Fiction (1994)": 5.0,
    "Fargo (1996)": 4.5,
    "Inception (2010)": 4.0,
    "Lord of the Rings: The Fellowship of the Ring, The (2001)": 4.0,
    "Titanic (1997)": 2.0,
    "Notebook, The (2004)": 1.5,
    "Twilight (2008)": 1.0,
    "Mamma Mia! (2008)": 2.0,
}
title_to_i = {t: i for i, t in enumerate(title)}
unknown = [t for t in MY_RATINGS if t not in title_to_i]
assert not unknown, f"Not found (use find_movie): {unknown}"
my_items = np.array([title_to_i[t] for t in MY_RATINGS]); my_r = np.array(list(MY_RATINGS.values()))
print(f"{len(MY_RATINGS)} ratings, your mean {my_r.mean():.2f} vs everyone's {ratings.rating.mean():.2f}")

## 1. The workhorse: ridge regression
Matrix factorisation predicts $\hat r_{ui} = \mu + b_u + b_i + p_u^\top q_i$. **Alternating least squares (ALS)** fixes all item vectors $q_i$ and solves for each user's $p_u$, then swaps. Each solve is ridge regression:
$$p_u = (Q_u^\top Q_u + \lambda I)^{-1} Q_u^\top r_u$$
where $Q_u$ stacks the vectors of the items user $u$ rated and $r_u$ holds their (bias-removed) ratings. Use `np.linalg.solve`, never `inv`.

In [ ]:
def solve_ridge(Q_sub, r_sub, lam):
    # TODO 1: return (Q_subᵀ Q_sub + lam·I)⁻¹ Q_subᵀ r_sub
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_ridge():
    rng = np.random.default_rng(0); Qs, rs = rng.normal(size=(30, 4)), rng.normal(size=30)
    ols = np.linalg.lstsq(Qs, rs, rcond=None)[0]
    return np.allclose(solve_ridge(Qs, rs, 0.0), ols) and np.linalg.norm(solve_ridge(Qs, rs, 100.0)) < np.linalg.norm(ols)
check("ridge solve", _t_ridge, "With lam=0 it must equal least squares; a large lam must shrink the solution.")

### Train ALS
Setup: 40 users are held out **completely** (they play "new users" in step 2); of the rest, a random 10 % of ratings is the test set.

In [ ]:
LAM_BIAS, K, LAM, ITERS = 5, 20, 20, 8
rng = np.random.default_rng(0)
cnt = ratings.groupby("u").size(); cold = rng.choice(cnt[cnt >= 60].index.to_numpy(), 40, replace=False)
warm = ratings[~ratings.u.isin(cold)]
is_test = rng.random(len(warm)) < 0.1; train, test = warm[~is_test], warm[is_test]
mu = train.rating.mean()

def fit_biases(df, iters=5):
    bu, bi = np.zeros(n_users), np.zeros(n_items)
    for _ in range(iters):
        bi = np.bincount(df.i, df.rating - mu - bu[df.u], n_items) / (np.bincount(df.i, minlength=n_items) + LAM_BIAS)
        bu = np.bincount(df.u, df.rating - mu - bi[df.i], n_users) / (np.bincount(df.u, minlength=n_users) + LAM_BIAS)
    return bu, bi

def groups(idx, n):   # row numbers grouped by user (or item)
    order = np.argsort(idx, kind="stable"); return np.split(order, np.cumsum(np.bincount(idx, minlength=n))[:-1])

rmse = lambda p, y: float(np.sqrt(np.mean((np.asarray(p) - np.asarray(y)) ** 2)))
b_user, b_item = fit_biases(train)
n_ratings = np.bincount(train.i, minlength=n_items)
print(f"test RMSE – global mean {rmse(np.full(len(test), mu), test.rating):.4f}, biases {rmse(mu + b_user[test.u] + b_item[test.i], test.rating):.4f}")

if ready("ridge solve"):
    u_, i_ = train.u.to_numpy(), train.i.to_numpy(); res = (train.rating - mu - b_user[u_] - b_item[i_]).to_numpy()
    by_u, by_i = groups(u_, n_users), groups(i_, n_items)
    P, Q = rng.normal(0, 0.1, (n_users, K)), rng.normal(0, 0.1, (n_items, K))
    for it in range(ITERS):
        for u in range(n_users):
            if len(by_u[u]): P[u] = solve_ridge(Q[i_[by_u[u]]], res[by_u[u]], LAM)
        for i in range(n_items):
            Q[i] = solve_ridge(P[u_[by_i[i]]], res[by_i[i]], LAM) if len(by_i[i]) else 0
    pred = mu + b_user[test.u] + b_item[test.i] + (P[test.u] * Q[test.i]).sum(1)
    als_rmse, bias_rmse = rmse(pred, test.rating), rmse(mu + b_user[test.u] + b_item[test.i], test.rating)
    print(f"test RMSE – ALS (k={K}) {als_rmse:.4f}")
    check("ALS beats biases", lambda: als_rmse < bias_rmse - 0.005, "ALS should beat the bias-only model on held-out ratings.")

## 2. Fold yourself in (no retraining)
A new user doesn't need a full retrain: keep the item vectors fixed and solve **one** ridge regression for the user. First the user bias $b_u = \frac{\sum (r - \mu - b_i)}{n + \lambda_b}$, then $p_u$ from the remaining residuals. We use a smaller $\lambda=2$ here (tuned on the cold users). Return `(b_user, p)`.

In [ ]:
def fold_in(items, ratings, lam=2.0):
    # TODO 2: user bias (shrunk with LAM_BIAS), then p = solve_ridge(Q[items], residual - b_user, lam)
    raise NotImplementedError  # TODO 2

In [ ]:
cold_eval = {}
if ready("ridge solve", "ALS beats biases"):
    err_b, err_mf, rho_b, rho_mf = [], [], [], []
    for u in cold:                                     # 20 known ratings → predict the rest
        d = ratings[ratings.u == u].sample(frac=1, random_state=0); known, rest = d.iloc[:20], d.iloc[20:]
        try: bu, p = fold_in(known.i.to_numpy(), known.rating.to_numpy())
        except NotImplementedError: break
        it, y = rest.i.to_numpy(), rest.rating.to_numpy()
        pb = mu + bu + b_item[it]; pm = pb + Q[it] @ p
        err_b += list((pb - y) ** 2); err_mf += list((pm - y) ** 2); rho_b.append(spearmanr(pb, y)[0]); rho_mf.append(spearmanr(pm, y)[0])
    else:
        cold_eval = dict(bias=np.sqrt(np.mean(err_b)), mf=np.sqrt(np.mean(err_mf)), rho_b=np.mean(rho_b), rho_mf=np.mean(rho_mf))
        print("40 new users, 20 ratings each:  RMSE biases {bias:.4f} → fold-in MF {mf:.4f}   ranking (Spearman) {rho_b:.3f} → {rho_mf:.3f}".format(**cold_eval))
def _t_fold():
    if not cold_eval: raise NotImplementedError
    return cold_eval["mf"] < cold_eval["bias"] and cold_eval["rho_mf"] > cold_eval["rho_b"]
check("fold-in", _t_fold,
      "Fold-in MF should beat biases-only for brand-new users on both RMSE and ranking.")

Honest result: the factors add a **small** gain over biases for new users with 20 ratings. Most of the win comes from the user's own bias (are you a harsh or a generous rater?). The ranking gain is what you feel as a user.

## 3. Your top 10
Score every movie, **remove the ones you rated**, and skip obscure films with fewer than `min_ratings` ratings (their item vectors are mostly noise). Return the indices of the top `n`.

In [ ]:
def recommend(b_user, p, seen, n=10, min_ratings=20):
    # TODO 3: scores = mu + b_item + b_user + Q @ p; mask seen and rarely-rated items with -inf; top n indices
    raise NotImplementedError  # TODO 3

In [ ]:
def _t_rec():
    top = recommend(0.0, np.zeros(K), seen={int(np.argmax(np.where(n_ratings >= 20, b_item, -9)))}, n=5)
    allowed = np.where(n_ratings >= 20, b_item, -np.inf); allowed[np.argmax(allowed)] = -np.inf
    return len(top) == 5 and list(top) == list(np.argsort(-allowed)[:5])
check("recommend", _t_rec, "With p = 0 the ranking is by item bias; the seen movie and movies with < 20 ratings must be excluded.")

if ready("fold-in", "recommend"):
    my_b, my_p = fold_in(my_items, my_r)
    top = recommend(my_b, my_p, seen=set(my_items))
    mf_scores = mu + b_item + my_b + Q @ my_p
    print(f"your bias {my_b:+.2f}\n")
    print(pd.DataFrame({"movie": title[top], "predicted ★": (mu + b_item + my_b + Q @ my_p)[top].round(2), "genres": movies.genres.to_numpy()[top]}).to_string(index=False))

## 4. Hybrid: add content
MF can't recommend a film nobody rated, and leans towards the mainstream. **Content** fixes that: describe each movie by its genres + user tags (TF-IDF), build your **profile** as the rating-weighted average of the movies you rated (ratings centred on *your* mean, so disliked films push the profile *away*), and score movies by cosine similarity. Return the L2-normalised profile.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tag_text = tags.groupby("movieId").tag.apply(lambda s: " ".join(s.astype(str).str.lower().str.replace(" ", "_")))
docs = (movies.genres.str.replace("|", " ").str.replace("-", "") + " " + movies.movieId.map(tag_text).fillna("")).to_numpy()
item_vecs = TfidfVectorizer(min_df=2).fit_transform(docs).toarray()
item_vecs /= np.linalg.norm(item_vecs, axis=1, keepdims=True) + 1e-12

def content_profile(items, ratings):
    # TODO 4: weights = ratings − mean(ratings); profile = Σ weight · item_vecs[item]; return profile / ‖profile‖
    raise NotImplementedError  # TODO 4

In [ ]:
def _t_prof():
    anim = np.where(movies.genres.str.contains("Animation") & (n_ratings >= 50))[0][:6]
    horror = np.where(movies.genres.str.contains("Horror") & (n_ratings >= 50))[0][:6]
    prof = content_profile(np.r_[anim, horror], np.r_[np.full(6, 5.0), np.full(6, 1.0)])
    s = item_vecs @ prof; s[n_ratings < 20] = -np.inf; top = np.argsort(-s)[:10]
    return abs(np.linalg.norm(prof) - 1) < 1e-6 and sum("Horror" in g for g in movies.genres.to_numpy()[top]) == 0 and \
        sum("Animation" in g for g in movies.genres.to_numpy()[top]) >= 7
check("content profile", _t_prof, "A profile that loves animation and hates horror should recommend ≥ 7 animated films and no horror.")

ALPHA = 0.7     # weight on MF; 1 − ALPHA on content
if ready("fold-in", "recommend", "content profile"):
    z = lambda x: (x - x[np.isfinite(x)].mean()) / x[np.isfinite(x)].std()
    content = item_vecs @ content_profile(my_items, my_r)
    hybrid = ALPHA * z(mf_scores) + (1 - ALPHA) * z(content)
    hybrid[list(my_items)] = -np.inf; hybrid[n_ratings < 5] = -np.inf      # content lets us go down to 5 ratings
    top_h = np.argsort(-hybrid)[:10]
    print(pd.DataFrame({"movie": title[top_h], "MF z": z(mf_scores)[top_h].round(2), "content z": z(content)[top_h].round(2),
                        "#ratings": n_ratings[top_h]}).to_string(index=False))

**Multimodal next step:** add poster images (CLIP embeddings) as a third signal, as in [[Project - MovieLens Recommender]] milestone 5 and [[Multimodal Recommender Systems]].

---
# Part 2 – Your Spotify history 🎧
**Get your data:** Spotify → *Account* → *Privacy settings* → *Download your data*. "Account data" arrives in ~5 days and contains `StreamingHistory_music_*.json`. "Extended streaming history" takes up to 30 days and contains `Streaming_History_Audio_*.json` covering *all* your years. Unzip the folder into `data/spotify/`. Both formats are supported.

Until then the notebook uses a **synthetic demo history** (clearly labelled), so every cell runs.

In [ ]:
SPOTIFY = pathlib.Path("data/spotify")

def load_spotify(folder):
    rows = []
    for f in sorted(folder.rglob("*.json")):
        if not re.match(r"(StreamingHistory|Streaming_History_Audio)", f.name): continue
        for e in json.loads(f.read_text(encoding="utf-8")):
            if "endTime" in e:   # account-data format
                rows.append((e["endTime"], e["artistName"], e["trackName"], e["msPlayed"]))
            elif e.get("master_metadata_album_artist_name"):   # extended format (podcasts have no artist)
                rows.append((e["ts"], e["master_metadata_album_artist_name"], e["master_metadata_track_name"], e["ms_played"]))
    df = pd.DataFrame(rows, columns=["ts", "artist", "track", "ms"])
    df["ts"] = pd.to_datetime(df.ts, utc=True)
    return df

def make_demo_history(seed=0):
    """SYNTHETIC listening history with taste clusters – only for trying the notebook before your export arrives."""
    r = np.random.default_rng(seed)
    clusters = {"dansk pop": ["Tobias Rahim", "Medina", "Christopher", "MØ", "Lukas Graham", "Gilli"],
                "dansk rock": ["Kim Larsen", "Gasolin'", "TV-2", "Sort Sol", "D-A-D", "Malurt"],
                "metal": ["Volbeat", "Metallica", "Iron Maiden", "Ghost", "Mastodon", "Gojira"],
                "electronic": ["Kölsch", "Trentemøller", "Daft Punk", "Bicep", "Fred again..", "Justice"],
                "jazz": ["Miles Davis", "John Coltrane", "Bill Evans", "Chet Baker", "Thelonious Monk", "Niels-Henning Ørsted Pedersen"]}
    taste = r.dirichlet(np.ones(len(clusters)) * 0.8); names = list(clusters)
    t, rows = pd.Timestamp("2025-01-01 08:00", tz="UTC"), []
    for _ in range(600):                                          # 600 sessions
        c = clusters[names[r.choice(len(names), p=taste)]]
        for _ in range(r.integers(4, 15)):
            a = c[r.integers(len(c))] if r.random() < 0.9 else r.choice(sum(clusters.values(), []))
            rows.append((t, a, f"{a} – track {r.integers(1, 30)}", int(r.integers(20_000, 240_000)))); t += pd.Timedelta(minutes=int(r.integers(2, 5)))
        t += pd.Timedelta(hours=int(r.integers(2, 30)))
    return pd.DataFrame(rows, columns=["ts", "artist", "track", "ms"])

history = load_spotify(SPOTIFY) if SPOTIFY.exists() and any(SPOTIFY.rglob("*.json")) else pd.DataFrame()
DEMO = history.empty
if DEMO: history = make_demo_history(); print("⚠️  Using the SYNTHETIC demo history – put your export in data/spotify/ for the real thing.")
history = history[history.ms >= 30_000].sort_values("ts").reset_index(drop=True)      # Spotify counts a stream after 30 s
print(f"{len(history):,} plays, {history.artist.nunique()} artists, {history.ts.min():%Y-%m-%d} → {history.ts.max():%Y-%m-%d}")
history.artist.value_counts().head(10)

## 5. Sessions
Songs played close together tell us which artists *go together* for you. Split the history into **sessions**: a new session starts when the gap since the previous play exceeds `gap_minutes`. Return an integer session id per row (in time order).

In [ ]:
def sessionize(df, gap_minutes=30):
    # TODO 5: sort by ts; a new session starts where the time difference to the previous play > gap; cumulative sum
    raise NotImplementedError  # TODO 5

In [ ]:
def _t_sess():
    ts = pd.to_datetime(["2025-01-01 10:00", "2025-01-01 10:04", "2025-01-01 11:00", "2025-01-01 11:10", "2025-01-02 09:00"], utc=True)
    return list(sessionize(pd.DataFrame({"ts": ts}), 30)) == [0, 0, 1, 1, 2]
check("sessions", _t_sess, "Gaps of 4 and 10 min stay in the session; 56 min and a day start new ones → [0, 0, 1, 1, 2].")

## 6. Artist embeddings from co-occurrence (word2vec without a neural net)
Count how often two artists appear in the same session ($C$), turn counts into **positive pointwise mutual information**
$$\text{PPMI}_{ij} = \max\Big(0, \log\frac{p(i,j)}{p(i)\,p(j)}\Big)$$
and factorise with SVD. Levy & Goldberg showed that word2vec's skip-gram with negative sampling implicitly factorises a (shifted) PMI matrix, so this is the same idea as [[Paper-to-Code Months]] month 2. Implement `ppmi(C)` (zeros where the count is 0).

In [ ]:
def ppmi(C):
    # TODO 6: p_ij = C / total, p_i = row sums / total, p_j = column sums / total; max(0, log(p_ij / (p_i p_j))), 0 where C == 0
    raise NotImplementedError  # TODO 6

In [ ]:
def _t_ppmi():
    C = np.array([[0, 4, 0], [4, 0, 1], [0, 1, 0]], float)
    out = ppmi(C); want = np.zeros((3, 3)); want[0, 1] = want[1, 0] = np.log(10 * 4 / (4 * 5)); want[1, 2] = want[2, 1] = np.log(10 * 1 / (5 * 1))
    return np.allclose(out, want)
check("PPMI", _t_ppmi, "For C = [[0,4,0],[4,0,1],[0,1,0]] (total 10) expect log 2 at (0,1) and log 2 at (1,2), zeros elsewhere.")

if ready("sessions", "PPMI"):
    history["session"] = sessionize(history)
    top_artists = history.artist.value_counts().index[:300]                      # keep it small and dense
    h = history[history.artist.isin(top_artists)]; a_idx = {a: i for i, a in enumerate(top_artists)}
    S = pd.crosstab(h.session, h.artist.map(a_idx)).clip(upper=1)                # session × artist (played or not)
    C = (S.T @ S).to_numpy().astype(float); np.fill_diagonal(C, 0)
    U_, s_, _ = np.linalg.svd(ppmi(C)); emb = U_[:, :16] * np.sqrt(s_[:16])
    emb /= np.linalg.norm(emb, axis=1, keepdims=True) + 1e-12

    def similar_artists(name, n=5):
        sims = emb @ emb[a_idx[name]]; sims[a_idx[name]] = -np.inf          # never recommend the artist itself
        return [top_artists[j] for j in np.argsort(-sims)[:n]]
    for a in top_artists[:5]: print(f"{a:>28} → {similar_artists(a)}")

In [ ]:
if ready("sessions", "PPMI") and DEMO:
    check("demo clusters recovered", lambda: set(similar_artists("Kim Larsen", 4)) <= {"Gasolin'", "TV-2", "Sort Sol", "D-A-D", "Malurt"},
          "In the demo data Kim Larsen's nearest artists should all be from the 'dansk rock' cluster.")

**Ideas for your real history:** "artists like X that I *rarely* play" (similar but low play count = discovery), plot the embeddings with t-SNE/UMAP coloured by year, find your *seasonal* artists (play counts by month), or score skip rates (`skipped` field in the extended export).

<details><summary>▶ Solution</summary>

```python
def solve_ridge(Q_sub, r_sub, lam):
    k = Q_sub.shape[1]
    return np.linalg.solve(Q_sub.T @ Q_sub + lam * np.eye(k), Q_sub.T @ r_sub)
```

```python
def fold_in(items, ratings, lam=2.0):
    resid = ratings - mu - b_item[items]
    b_user = resid.sum() / (len(items) + LAM_BIAS)
    p = solve_ridge(Q[items], resid - b_user, lam)
    return b_user, p
```

```python
def recommend(b_user, p, seen, n=10, min_ratings=20):
    scores = mu + b_item + b_user + Q @ p
    scores[n_ratings < min_ratings] = -np.inf
    scores[list(seen)] = -np.inf
    return np.argsort(-scores)[:n]
```

```python
def content_profile(items, ratings):
    w = ratings - ratings.mean()
    profile = (w[:, None] * item_vecs[items]).sum(0)
    return profile / (np.linalg.norm(profile) + 1e-12)
```

```python
def sessionize(df, gap_minutes=30):
    df = df.sort_values("ts").reset_index(drop=True)
    new = df.ts.diff() > pd.Timedelta(minutes=gap_minutes)
    return new.cumsum().to_numpy()
```

```python
def ppmi(C):
    total = C.sum()
    p_ij = C / total
    p_i = C.sum(1, keepdims=True) / total
    p_j = C.sum(0, keepdims=True) / total
    with np.errstate(divide="ignore", invalid="ignore"):
        pmi = np.log(p_ij / (p_i * p_j))
    return np.where(np.isfinite(pmi) & (pmi > 0), pmi, 0.0)
```
</details>

## Make it yours (stretch goals)
- **Recipes instead of movies:** the [Food.com recipes and interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions) dataset (Kaggle, free account) has ratings + ingredients. Same code: ALS on ratings, content = ingredients.
- **Implicit feedback:** treat your Spotify play counts as confidence and run implicit ALS (Hu, Koren & Volinsky 2008) to recommend *tracks*.
- **Evaluate yourself:** hide 5 of your own ratings, fold in with the rest, and see whether the hidden ones rank high.
- **Explain a recommendation:** which of your rated movies has the largest $q_i^\top q_j$ with the recommended one? "Because you liked …"